# Prueba de las funcionalidades nuevas

Este notebook ejercita, en orden, lo que agregamos al framework:

1. **Registros perezosos** — `ModelFactory` / `BenchmarkFactory`.
2. **Contrato de capacidades** — `Capabilities`, `check_compatibility`,
   `Model.requirements()` / `BenchMark.capabilities()`.
3. **Validacion en `run_episode`** — falla temprano si el benchmark no ofrece lo
   que el modelo requiere.
4. **Clase `Interface`** — `setup_scene()` (ver escena + pedir prompt) y `step()`.
5. **Serving + `RemoteModel`** — *(solo cluster)* servir un modelo por HTTP.

> **Entornos.** Correr con el env conda **`tipico`**. Las secciones 1-4 corren en
> local (no necesitan GPU ni mujoco). La seccion 5 (serving pi0/openvla) corre en
> el **cluster**; aqui queda como referencia y se auto-omite si no hay servidor.


In [ ]:
import sys, os
# Asegura que la raiz del repo este importable (el notebook vive en la raiz).
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("cwd:", ROOT)


## 1. Registros perezosos (`ModelFactory` / `BenchmarkFactory`)

Registrar y listar por nombre **no** importa los stacks pesados (torch/lerobot):
solo se importan al `create()` de ese modelo, en su propio entorno.

In [ ]:
from models import ModelFactory
from benchmarks import BenchmarkFactory

print("Modelos    registrados:", ModelFactory.available())
print("Benchmarks registrados:", BenchmarkFactory.available())

pesados = [m for m in ("torch", "lerobot", "transformers", "libero", "robosuite")
           if m in sys.modules]
print("Stacks pesados importados tras registrar:", pesados or "ninguno")

# 'random' no necesita GPU -> se puede crear localmente.
random_model = ModelFactory.create("random")
print("Creado:", type(random_model).__name__, "| torch importado?", "torch" in sys.modules)


## 2. Contrato de capacidades

`Capabilities` describe lo que un modelo **requiere** o un benchmark **ofrece**.
`check_compatibility` valida y, si falta algo, lanza `IncompatibleCapabilities`
con un mensaje *requiere / ofrece / falta*.

In [ ]:
from core import Capabilities, View, check_compatibility
from core.capabilities import IncompatibleCapabilities

req = Capabilities.of(views=[View.AGENT],
                      state=["eef_pos", "eef_quat", "gripper_qpos"], instruction=True)
libero  = Capabilities.of(views=[View.AGENT],
                          state=["eef_pos", "eef_quat", "gripper_qpos"], instruction=True)
example = Capabilities.of(views=[View.SIDE_1, View.SIDE_2, View.WRIST],
                          state=["joint_positions", "tcp"], instruction=True)

print("Requiere (pi0-like):", req.describe())
check_compatibility(req, libero)          # no lanza
print("Compatible con LIBERO: OK")

try:
    check_compatibility(req, example, model_name="pi0", benchmark_name="Example")
except IncompatibleCapabilities as e:
    print("\nIncompatible con Example (esperado):\n")
    print(e)


In [ ]:
import numpy as np
from core import Observation

# Red de seguridad: capacidades derivadas de una observacion REAL.
obs = Observation(images={View.AGENT.value: np.zeros((8, 8, 3), np.uint8)},
                  state={"eef_pos": np.zeros(3)}, instruction="pick")
print("Capacidades de la observacion:", Capabilities.from_observation(obs).describe())


### 2b. `requirements()` de los modelos reales (sin cargar pesos)

Los controladores importan torch de forma **perezosa** (dentro de `_load`), asi
que podemos leer lo que declaran sin GPU: instanciamos sin `__init__` y solo
llamamos a `requirements()`.

In [ ]:
from models.OpenVLA.openvla_controller import OpenVLAController
from models.Pi_zero.Pi_zero_controller import PiZeroController

def sin_cargar(cls, **attrs):
    """Instancia sin __init__ (evita _load/torch); solo para leer requirements()."""
    obj = object.__new__(cls)
    for k, v in attrs.items():
        setattr(obj, k, v)
    return obj

vla = sin_cargar(OpenVLAController, view=View.AGENT)
pi0 = sin_cargar(PiZeroController, view=View.AGENT, wrist_view=View.WRIST)
print("OpenVLA requiere:", vla.requirements().describe())
print("pi0     requiere:", pi0.requirements().describe())
print("torch importado?", "torch" in sys.modules)


## 3. Validacion automatica en `run_episode`

`run_episode` compara `model.requirements()` con `benchmark.capabilities()` justo
tras el `reset()`. Usamos dobles ligeros (sin mujoco/torch) para verificar los dos
caminos.

In [ ]:
from core import Action, StepResult, run_episode

class FakeBench:
    """Benchmark de juguete: declara sus capacidades y termina tras N pasos."""
    def __init__(self, caps, done_after=3):
        self._caps, self._done_after, self._n = caps, done_after, 0
    def capabilities(self):
        return self._caps
    def _obs(self):
        return Observation(
            images={View.AGENT.value: np.zeros((8, 8, 3), np.uint8)},
            state={"eef_pos": np.zeros(3), "eef_quat": np.array([0, 0, 0, 1.0]),
                   "gripper_qpos": np.zeros(2)}, instruction="demo")
    def reset(self, episode=None):
        self._n = 0
        return self._obs()
    def step(self, action):
        self._n += 1
        return StepResult(self._obs(), reward=1.0, done=self._n >= self._done_after)
    def close(self):
        pass

class NeedsAgentview:
    """Modelo de juguete que requiere la vista agentview."""
    def reset(self):
        pass
    def requirements(self):
        return Capabilities.of(views=[View.AGENT], instruction=True)
    def act(self, observation):
        return [Action.from_cartesian(np.zeros(6), gripper=1.0)]

# a) contrato cumplido -> corre
res = run_episode(FakeBench(libero), NeedsAgentview(), max_steps=5)
print("[a] compatible -> success:", res.success, "| steps:", res.steps)

# b) contrato incumplido -> falla ANTES del primer step
try:
    run_episode(FakeBench(Capabilities.of(views=[View.SIDE_1])), NeedsAgentview())
except IncompatibleCapabilities as e:
    print("\n[b] incompatible (esperado):\n")
    print(e)


## 4. Clase `Interface`

Empareja un benchmark y un modelo con dos operaciones:
- `setup_scene()` — reinicia, **muestra el estado inicial** y **pide el prompt**.
- `step()` — observa (con el prompt inyectado), pide el chunk y lo ejecuta.

Para el notebook usamos una subclase que **fija** el prompt (sin bloquear con
`input()`); mas abajo se muestra el uso interactivo real.

In [ ]:
from core import Interface

class PatternBench(FakeBench):
    """FakeBench con una imagen con patron, para ver algo en _show()."""
    def _obs(self):
        img = np.zeros((64, 64, 3), np.uint8)
        img[:32, :, 0] = 200      # mitad superior roja
        img[32:, :, 2] = 200      # mitad inferior azul
        return Observation(images={View.AGENT.value: img},
                           state={"eef_pos": np.zeros(3)}, instruction="demo")

class InterfaceDemo(Interface):
    """Version scripteada: fija el prompt en vez de pedirlo por teclado."""
    def __init__(self, benchmark, model, prompt):
        super().__init__(benchmark, model)
        self._fake_prompt = prompt
    def _ask_prompt(self):
        print(f'(prompt fijado por el demo: "{self._fake_prompt}")')
        return self._fake_prompt

demo = InterfaceDemo(PatternBench(libero, done_after=2), NeedsAgentview(),
                     prompt="pick up the red cube")
_ = demo.setup_scene()      # muestra la imagen inicial y fija el prompt


In [ ]:
result = demo.step()        # observa con el prompt, pide el chunk y lo ejecuta
print("prompt que recibio el modelo -> instruction:", repr(demo._observation.instruction))
print("done:", result.done, "| reward:", result.reward)


**Uso interactivo real** (fuera de este demo scripteado): con un benchmark y un
modelo de verdad, simplemente:

```python
from core import Interface
from benchmarks import BenchmarkFactory
from models.serving import RemoteModel

iface = Interface(BenchmarkFactory.create("libero", task_id=0),
                  RemoteModel(url="http://localhost:9000"))
iface.setup_scene()   # abre la imagen y pregunta: "Prompt para el modelo: "
iface.step()          # ejecuta el chunk del modelo
```


## 5. Serving + `RemoteModel`  *(CLUSTER)*

Los modelos reales (pi0/openvla) corren en el **cluster**, servidos por HTTP.
Endpoints: `GET /health`, `GET /capabilities`, `POST /reset`, `POST /act`,
`POST /context`. Esta celda se auto-omite si no hay servidor accesible.

In [ ]:
import urllib.request

URL = "http://localhost:9000"   # 9000=pi0, 9001=openvla (via tunel SSH al nodo)

def servidor_vivo(url, timeout=1.0):
    try:
        with urllib.request.urlopen(url + "/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

if servidor_vivo(URL):
    from models.serving import RemoteModel
    model = RemoteModel(url=URL)
    # /capabilities cruza la frontera de proceso -> requirements() del modelo remoto:
    print("Requisitos del modelo remoto:", model.requirements().describe())
    print("Listo: ya puedes usarlo con Interface o run_experiments.")
else:
    print("No hay servidor en", URL, "-> seccion CLUSTER omitida.\n")
    print("En el cluster:  sbatch scripts/experiment.sh   (elige MODEL=pi0|openvla)")
    print("En tu maquina:  ssh -L localhost:9000:<nodo>:9000 kraken   # tunel al servidor")


---
### Resumen

Si las secciones 1-4 corrieron sin error, las funcionalidades nuevas
(registros, contrato de capacidades, validacion en `run_episode` e `Interface`)
funcionan. La seccion 5 se valida contra un servidor real en el cluster.